In [ ]:
!pip install -q apify-client sentence-transformers openpyxl pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.4/140.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from apify_client import ApifyClient
from sentence_transformers import SentenceTransformer, util
API_TOKEN = "Input APIfy Token"
ACTOR_ID = "LpVuK3Zozwuipa5bp"
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
#Ideal Candidate Profile (ICP)

ICP = """
Recent 2025 or 2026 graduates in Civil Engineering,
Mechanical Engineering, Structural Engineering,
Construction Management, or Project Management.

Located in the United States.

Entry-level with internship experience.

Experience using AutoCAD.

Potential interest in Telecom or OSP engineering roles.
"""
icp_embedding = model.encode(ICP, convert_to_tensor=True)

In [ ]:
# Data and API

df = pd.read_csv("leads.csv")
urls = df["profile_url"].tolist()
client = ApifyClient(API_TOKEN)

run_input = {"profileScraperMode": "Profile details no email ($4 per 1k)",
    "queries": urls}

run = client.actor(ACTOR_ID).call(run_input=run_input)

# Get dataset items
profiles = list(client.dataset(
        run.default_dataset_id).iterate_items())

print(f"Profiles scraped: {len(profiles)}")


[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> Status: RUNNING, Message: 
[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> 2026-06-29T05:58:07.835Z ACTOR: Pulling container image of build kv0Sz4HLDJUfEJ0yg from registry.
[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> 2026-06-29T05:58:07.837Z ACTOR: Creating container.
[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> 2026-06-29T05:58:07.969Z ACTOR: Starting container.
[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> 2026-06-29T05:58:07.970Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> 2026-06-29T05:58:08.859Z INFO  System info {"apifyVersion":"3.7.2","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v20.20.2"}
[apify.linkedin-profile-scraper runId:QC5rcAMQow7mYX0Wa] -> 2026-06-29T05:58:12.706Z Starting item#1 {"query":"https://www.linkedin.com/in/james-larkin25/"}...
[apify.linkedin-

Profiles scraped: 5


In [ ]:
#Profile Text

def build_profile_text(profile):

    headline = profile.get("headline", "")

    about = profile.get("about", "")

    location = str(profile.get("location", ""))

    education_text = ""

    for edu in profile.get("education", []):
        education_text += (str(edu) + " ")

    experience_text = ""

    for exp in profile.get("experience", []):
        experience_text += (str(exp) + " ")

    skills_text = ""

    for skill in profile.get("skills", []):
        skills_text += (str(skill) + " ")

    return f"""
    Location:
    {location}

    Headline:
    {headline}

    About:
    {about}

    Education:
    {education_text}

    Experience:
    {experience_text}

    Skills:
    {skills_text}
    """

In [ ]:
# Scoring profile

def score_profile(profile):

    profile_text = build_profile_text(profile)

    profile_embedding = model.encode(profile_text,
        convert_to_tensor=True)

    similarity = util.cos_sim(profile_embedding,
        icp_embedding).item()

    semantic_score = int((similarity + 1) * 50)
    location_obj = profile.get("location", {})
    if isinstance(location_obj, dict):
      location_text = location_obj.get(
        "linkedinText","").lower()
    else:
      location_text = str(location_obj).lower()

    location_bonus = 0

    if ("united states"
        in location_text
        or "usa"
        in location_text):
        location_bonus = 20

    final_score = (semantic_score * 0.8) + location_bonus

    final_score = round(min(final_score, 100))

    return final_score

In [ ]:
#Justification

def generate_justification(profile, score):

    location_obj = profile.get("location", {})

    if isinstance(location_obj, dict):
        location = location_obj.get("linkedinText","Unknown Location")
    else:
        location = str(location_obj)
    headline = profile.get("headline","No headline available")

    if score >= 75:
        assessment = ("The profile shows strong alignment "
            "with the target engineering candidate profile.")

    elif score >= 60:

        assessment = ("The profile shows moderate alignment "
            "with the target engineering candidate profile.")
    else:
        assessment = ("The profile does not sufficiently align "
            "with the target engineering candidate profile.")

    return (f"Score {score}/100. "
        f"Candidate location: {location}. "
        f"Professional background: {headline}. "
        f"{assessment}")


In [ ]:
# Email to send

def generate_email(profile, score):
    if score < 75:
        return ""

    name = (profile.get("firstName", "")+ " "
        + profile.get("lastName", "")).strip()

    headline = profile.get("headline", "")
    location = str(profile.get("location", ""))

    return f"""
Hi {name},

I came across your LinkedIn profile and noticed your background as {headline}.

Your profile and experience appear closely aligned with the type of talent we are currently seeking.

I would love to connect and discuss potential opportunities with you.

Best regards,
Skarion HR Team
"""


In [ ]:
#Provessing

results = []

for profile in profiles:

    score = score_profile(profile)

    name = (profile.get("firstName", "")+ " "
        + profile.get("lastName", "")).strip()

    email_text = generate_email(profile,score)

    if score >= 75:
        status = "Pending"
    else:
        status = "Rejected"

    results.append({"Candidate Name":name, "Profile Link":
        profile.get(
            "linkedinUrl",""),

        "Fit Score":
        score,

        "Justification":
        generate_justification(profile,score),

        "Generated Personalized Outreach Text":
        email_text,

        "Status":
        status})

In [ ]:
# Email searching for only pending status

pending_urls = [
    row["Profile Link"]
    for row in results
    if row["Status"] == "Pending"]

print(f"Searching emails for {len(pending_urls)} qualified candidates...")

email_lookup = {}

if len(pending_urls) > 0:

    run_input = {
        "profileScraperMode": "Profile details + email search ($10 per 1k)",
        "queries": pending_urls}

    run = client.actor(ACTOR_ID).call(run_input=run_input)

    email_profiles = list(client.dataset(run.default_dataset_id).iterate_items())

    for profile in email_profiles:

        linkedin_url = profile.get("linkedinUrl", "")

        email = ""

        # Try the most common fields
        if profile.get("emails"):
            emails = profile["emails"]

            if isinstance(emails, list) and len(emails) > 0:

                if isinstance(emails[0], dict):
                    email = emails[0].get("email", "")
                else:
                    email = str(emails[0])

        elif profile.get("workEmail"):
            email = profile["workEmail"]

        elif profile.get("verifiedEmail"):
            email = profile["verifiedEmail"]

        email_lookup[linkedin_url] = email

print("Email search completed.")

Searching emails for 3 qualified candidates...


[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> Status: RUNNING, Message: 
[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> 2026-06-29T05:58:41.121Z ACTOR: Pulling container image of build kv0Sz4HLDJUfEJ0yg from registry.
[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> 2026-06-29T05:58:41.123Z ACTOR: Creating container.
[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> 2026-06-29T05:58:41.166Z ACTOR: Starting container.
[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> 2026-06-29T05:58:41.167Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> 2026-06-29T05:58:42.321Z INFO  System info {"apifyVersion":"3.7.2","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v20.20.2"}
[apify.linkedin-profile-scraper runId:HnFLfsTNBodzBwtpk] -> 2026-06-29T05:58:46.149Z Starting item#1 {"query":"https://www.linkedin.com/in/james-larkin25"}...
[apify.linkedin-p

Email search completed.


In [ ]:
#Export

for row in results:

    row["Candidate Email"] = email_lookup.get(row["Profile Link"], "")

output_df = pd.DataFrame(results)

column_order = ["Candidate Name",
    "Profile Link",
    "Candidate Email",
    "Fit Score",
    "Justification",
    "Generated Personalized Outreach Text",
    "Status"]

output_df = output_df[column_order]

output_df.to_excel("qualified_leads.xlsx",index=False)

print("qualified_leads.xlsx created.")

qualified_leads.xlsx created.
